# GreenEye: GitHub Public Notebook

This notebook is the **GitHub-ready public version** of the GreenEye implementation.

It is designed to work with the following repository structure:

```text
greeneye/
├── GreenEye.ipynb
└── data/
    └── dataset_traffic/
        ├── data.yaml
        ├── train.txt
        ├── val.txt
        ├── images/
        └── labels/
```

Run the notebook **from top to bottom**. A **GPU runtime** is recommended.

## 1. Environment Setup

In [ ]:
%cd /content/
!git clone https://github.com/ultralytics/yolov5.git
%cd /content/yolov5/
!pip install -r requirements.txt
!pip install pyyaml==5.4.1

## 2. Repository Path Configuration

In [ ]:
import os
from pathlib import Path

REPO_ROOT = Path(os.environ.get("GREENEYE_REPO_ROOT", "/content/greeneye"))
DATA_ROOT = REPO_ROOT / "data" / "dataset_traffic"
IMAGES_DIR = DATA_ROOT / "images"
LABELS_DIR = DATA_ROOT / "labels"
DATA_YAML = DATA_ROOT / "data.yaml"
TRAIN_TXT = DATA_ROOT / "train.txt"
VAL_TXT = DATA_ROOT / "val.txt"
WEIGHTS_PATH = Path(os.environ.get("GREENEYE_WEIGHTS", str(REPO_ROOT / "weights" / "Balanced_1113_9_1_best.pt")))

print("REPO_ROOT   :", REPO_ROOT)
print("DATA_ROOT   :", DATA_ROOT)
print("IMAGES_DIR  :", IMAGES_DIR)
print("LABELS_DIR  :", LABELS_DIR)
print("DATA_YAML   :", DATA_YAML)
print("TRAIN_TXT   :", TRAIN_TXT)
print("VAL_TXT     :", VAL_TXT)
print("WEIGHTS_PATH:", WEIGHTS_PATH)

## 3. Normalize Dataset Paths

In [ ]:
from pathlib import Path
import yaml

def normalize_split_file(txt_path, images_dir):
    if not txt_path.exists():
        print(f"Missing: {txt_path}")
        return
    lines = [line.strip() for line in txt_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    normalized = []
    for line in lines:
        name = Path(line).name
        normalized.append(str(images_dir / name))
    txt_path.write_text("
".join(normalized) + "
", encoding="utf-8")
    print(f"Normalized {txt_path.name}: {len(normalized)} entries")

if TRAIN_TXT.exists():
    normalize_split_file(TRAIN_TXT, IMAGES_DIR)
if VAL_TXT.exists():
    normalize_split_file(VAL_TXT, IMAGES_DIR)

if DATA_YAML.exists():
    data = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
    data["train"] = str(TRAIN_TXT)
    data["val"] = str(VAL_TXT)
    DATA_YAML.write_text(yaml.safe_dump(data, sort_keys=False), encoding="utf-8")
    print("Updated data.yaml")
    print(data)
else:
    print("Missing data.yaml")

## 4. Dataset Inspection

In [ ]:
from glob import glob

jpgs = glob(str(IMAGES_DIR / "*.JPG")) + glob(str(IMAGES_DIR / "*.jpg")) + glob(str(IMAGES_DIR / "*.png"))
labels = glob(str(LABELS_DIR / "*.txt"))

print("Number of images:", len(jpgs))
print("Number of labels:", len(labels))

if DATA_YAML.exists():
    print("
--- data.yaml ---")
    print(DATA_YAML.read_text(encoding="utf-8"))

if TRAIN_TXT.exists():
    print("
--- train.txt (first 5) ---")
    print("
".join(TRAIN_TXT.read_text(encoding="utf-8").splitlines()[:5]))

if VAL_TXT.exists():
    print("
--- val.txt (first 5) ---")
    print("
".join(VAL_TXT.read_text(encoding="utf-8").splitlines()[:5]))

## 5. Optional Sample Split Generation

In [ ]:
# Use this only if you want to regenerate train/val split from the images folder.
# If you already uploaded curated train.txt and val.txt, you can skip this cell.

from sklearn.model_selection import train_test_split
from glob import glob

img_list = sorted(glob(str(IMAGES_DIR / "*.JPG")) + glob(str(IMAGES_DIR / "*.jpg")) + glob(str(IMAGES_DIR / "*.png")))
if len(img_list) >= 2:
    train_img_list, val_img_list = train_test_split(img_list, test_size=0.2, random_state=2000)
    TRAIN_TXT.write_text("
".join(train_img_list) + "
", encoding="utf-8")
    VAL_TXT.write_text("
".join(val_img_list) + "
", encoding="utf-8")
    print("Generated train/val split")
    print("Train images:", len(train_img_list))
    print("Validation images:", len(val_img_list))
else:
    print("Not enough images to generate a split. Upload images first.")

## 6. Model Training

In [ ]:
%cd /content/yolov5/
!python train.py --img 640 --batch 16 --epochs 30 --data "{DATA_YAML}" --cfg ./models/yolov5s.yaml --weights yolov5s.pt --name greeneye_result

## 7. Example Inference

In [ ]:
from glob import glob

img_list = sorted(glob(str(IMAGES_DIR / "*.JPG")) + glob(str(IMAGES_DIR / "*.jpg")) + glob(str(IMAGES_DIR / "*.png")))
if len(img_list) == 0:
    print("No images found in", IMAGES_DIR)
else:
    val_img_path = img_list[0]
    print("Inference source:", val_img_path)
    !python detect.py --weights "{WEIGHTS_PATH}" --img 640 --conf 0.2 --source "{val_img_path}"

# Example for video inference:
# !python detect.py --weights "{WEIGHTS_PATH}" --img 640 --conf 0.4 --source "/content/sample_video.mp4"